In [7]:
import pandas as pd
import numpy as np
import tensorflow as tf
from keras.layers import Dense,Dropout
from keras.models import Sequential
import keras_tuner as kt
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

In [11]:
data=pd.read_csv('data/huge_1M_titanic.csv')

In [3]:
data.shape

(1000000, 12)

In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 12 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   PassengerId  1000000 non-null  int64  
 1   Survived     1000000 non-null  int64  
 2   Pclass       1000000 non-null  int64  
 3   Name         1000000 non-null  object 
 4   Sex          1000000 non-null  object 
 5   Age          801400 non-null   float64
 6   SibSp        1000000 non-null  int64  
 7   Parch        1000000 non-null  int64  
 8   Ticket       1000000 non-null  object 
 9   Fare         1000000 non-null  float64
 10  Cabin        229805 non-null   object 
 11  Embarked     997760 non-null   object 
dtypes: float64(2), int64(5), object(5)
memory usage: 91.6+ MB


In [12]:
data = data.drop(columns=['PassengerId','Name','Age','Ticket','Cabin'])

In [13]:
data['Embarked'] = data['Embarked'].replace({'S': 'Southampton','C': 'Cherbourg', 'Q': 'Queenstown'})

In [14]:
data.dropna(subset=['Embarked'], inplace=True)

In [17]:
le = LabelEncoder()

In [18]:
data['Sex'] = le.fit_transform(data['Sex'])

In [19]:
onehot = OneHotEncoder(sparse_output=False)


In [20]:
Embarked = onehot.fit_transform(data[['Embarked']])

In [21]:
Embarked = pd.DataFrame(Embarked, columns=onehot.get_feature_names_out(['Embarked']))

In [22]:
data = pd.concat([data.drop(columns=['Embarked']), Embarked], axis=1)

In [23]:
from sklearn.preprocessing import StandardScaler

In [24]:
scale = StandardScaler()

In [25]:
num_cols = ['Pclass','SibSp', 'Parch', 'Fare']

In [26]:
data[num_cols] = scale.fit_transform(data[num_cols])

In [27]:
data.shape

(999994, 9)

In [28]:
data.isnull().sum()

Survived                2234
Pclass                  2234
Sex                     2234
SibSp                   2234
Parch                   2234
Fare                    2234
Embarked_Cherbourg      2234
Embarked_Queenstown     2234
Embarked_Southampton    2234
dtype: int64

In [29]:
data = data.dropna()

In [30]:
data.isnull().sum()

Survived                0
Pclass                  0
Sex                     0
SibSp                   0
Parch                   0
Fare                    0
Embarked_Cherbourg      0
Embarked_Queenstown     0
Embarked_Southampton    0
dtype: int64

In [31]:
X = data.drop(columns=['Survived'])
y = data['Survived']

In [32]:
from sklearn.model_selection import train_test_split

In [33]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=20,random_state=42)  

In [34]:
X_train, X_valid, y_train, y_valid = train_test_split(X_train,y_train,test_size=0.2,random_state=42)    

In [35]:
import tensorflow as tf
from tensorflow import keras
from keras.models import Sequential
from keras.layers import Dense
from keras.callbacks import EarlyStopping, TensorBoard

print('TensorFlow:', tf.__version__)

TensorFlow: 2.18.0


In [27]:
# model = Sequential([Dense(64, activation='relu', input_shape=(X_train.shape[1],)), ## First layer with input shape
#             Dense(32, activation='relu'), ## Second layer
#             Dense(1, activation='sigmoid')]) ## Output layer with sigmoid activation for binary classification


In [28]:
# import tensorflow as tf

# opt = tf.keras.optimizers.Adam(learning_rate=0.01)

In [29]:
# model.compile(optimizer=opt, loss='binary_crossentropy', metrics=['accuracy'])

In [30]:
# model.fit(X_train, y_train, epochs=50)

## Optimal Optimizer
SGD, Adam, Adagrad, RMSprop, Adadelta, Adamax, Nadam, Ftrl

In [31]:
def build_model(hp):
    model = Sequential([Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
                       Dense(32, activation='relu'),
                       Dense(1, activation='sigmoid')])
    
    optimizer = hp.Choice('optimizer', values=['adam', 'sgd', 'rmsprop'])
    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    
    return model

In [32]:
tuner = kt.RandomSearch(build_model,max_trials=5, objective='val_accuracy', directory='my_dir', project_name='titanic_tuning')

Reloading Tuner from my_dir/titanic_tuning/tuner0.json


I0000 00:00:1776448802.429786  148298 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3524 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 6GB Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


In [33]:
tuner.search(X_train, y_train, epochs=10, validation_data=(X_valid, y_valid))

In [34]:
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'adam'}

In [35]:
model = tuner.get_best_models(num_models=1)#[0].evaluate(X_test, y_test)

/home/dharmender/anaconda3/lib/python3.10/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/home/dharmender/anaconda3/lib/python3.10/site-packages/keras/src/saving/saving_lib.py:719: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [36]:
model.fit(X_train, y_train,validation_data=(X_test, y_test), epochs=30,initial_epoch=11)

AttributeError: 'list' object has no attribute 'fit'

### Choose the Most Ideal Number of Nodes we should have in the Hidden Layer

In [44]:
import tensorflow as tf
tf.config.run_functions_eagerly(True)
from keras.layers import Input

In [45]:
opt = tf.keras.optimizers.Adam(learning_rate=0.01)

In [47]:
def build_model(hp):
    nodes = hp.Int('nodes',8,128,step=8)
    # activation_func = hp.Choice('activation', values=['relu', 'tanh'])

    model = Sequential()
    model.add(Input(shape=(X_train.shape[1],)))
    model.add(Dense(nodes, activation='relu'))
    model.add(Dense(nodes, activation='relu'))
    model.add(Dense(1, activation='sigmoid'))

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

In [42]:
tuner = kt.RandomSearch(build_model, objective='val_accuracy', max_trials=5, directory='my_dir', project_name='titanic_tuning')

Reloading Tuner from my_dir/titanic_tuning/tuner0.json


In [40]:
tuner.search(X_train, y_train, validation_data=(X_test, y_test), epochs=10)

In [48]:
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'adam'}

In [49]:
model = tuner.get_best_models(num_models=1)[0]

/home/dharmender/anaconda3/lib/python3.10/site-packages/keras/src/saving/saving_lib.py:719: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


ValueError: A total of 3 objects could not be loaded. Example error message for object <Dense name=dense, built=True>:

The shape of the target variable and the shape of the target value in `variable.assign(value)` must match. variable.shape=(8, 8), Received: value.shape=(8, 64). Target variable: <KerasVariable shape=(8, 8), dtype=float32, path=sequential/dense/kernel>

List of objects that could not be loaded:
[<Dense name=dense, built=True>, <Dense name=dense_1, built=True>, <Dense name=dense_2, built=True>]

In [ ]:
model.fit(X_train, y_train,validation_data=(X_valid, y_valid), epochs=50, initial_epoch=11) 

### Deciding Optimal Number of Hidden Layers with Tuning


In [1]:
from keras.layers import Dropout, Input, Dense

In [55]:
def build_model(hp):
    model = Sequential()
    model.add(Input(shape=(X_train.shape[1],)))
    
    for i in range(hp.Int('hidden',min_value=1, max_value=10, step=1)):
        #  model.add(88, activation='relu')
        model.add(Dense(hp.Int(f'nodes_{i}', 8, 128, step=8), activation='relu'))
    
    model.add(Dense(1, activation='sigmoid'))
    
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

In [56]:
tuner = kt.RandomSearch(build_model, objective='val_accuracy', max_trials=3, directory='my_dir', project_name='titanic_tuning')

Reloading Tuner from my_dir/titanic_tuning/tuner0.json


In [57]:
tuner.search(X_train, y_train, validation_data=(X_valid, y_valid), epochs=10)

In [58]:
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'adam'}

In [59]:
model = tuner.get_best_models(num_models=1)[0]

/home/dharmender/anaconda3/lib/python3.10/site-packages/keras/src/saving/saving_lib.py:719: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


ValueError: A total of 2 objects could not be loaded. Example error message for object <Dense name=dense, built=True>:

The shape of the target variable and the shape of the target value in `variable.assign(value)` must match. variable.shape=(8, 8), Received: value.shape=(8, 64). Target variable: <KerasVariable shape=(8, 8), dtype=float32, path=sequential/dense/kernel>

List of objects that could not be loaded:
[<Dense name=dense, built=True>, <Dense name=dense_1, built=True>]

## Hyperparameter Tuning : 
1. No. of Hidden Layers
2. No. of Nodes in each Hidden Layer
3. Optimal Optimizer
4. Dropout Layer -value


In [61]:
from keras.layers import Dropout, Input, Dense

In [70]:
def build_model(hp):
    model = Sequential()
    model.add(Input(shape=(X_train.shape[1],)))
    
    for i in range(hp.Int('num_layers', 1, 5,1)):
        nodes = hp.Int('nodes',min_value=8, max_value=128, step=8)
        model.add(Dense(nodes, activation='relu'))
        dropout_value = hp.Float('dropout', 0.1, 0.5, step=0.1)
        model.add(Dropout(dropout_value))

    model.add(Dense(1, activation='sigmoid'))
    optimizers = hp.Choice('optimizer', values=['SGD','RMSprop','adam','AdamW','Adadelta','Adagrad','Adamax','Nadam'])
    model.compile(optimizer=optimizers, loss='binary_crossentropy', metrics=['accuracy'])
    
    return model

In [74]:
tuner = kt.RandomSearch(build_model, objective='val_accuracy', max_trials=3, directory='all_in_onesd', project_name='titanic_tuning')

In [75]:
tuner.get_best_hyperparameters()[0].values

IndexError: list index out of range

In [76]:
tuner.get_best_models(num_models=1)[0]

IndexError: list index out of range

In [ ]:
model.fit(X_train, y_train, validation_data=(X_valid, y_valid), epochs=50, initial_epoch=11)